<a href="https://colab.research.google.com/github/elangbijak4/multi-agent-AI/blob/main/Hyperparameter_Tuning6_Agent_dengan_RF_dan_BLCAKBOARD_MEMORY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# INTEGRATED INTELLIGENT GRID AGENT
# Supervised Brain + RL + Blackboard + Meta-Learning
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
from collections import deque, defaultdict
from sklearn.tree import DecisionTreeClassifier

# ============================================================
# FEATURE TOGGLES (FOR TEACHING)
# ============================================================
USE_RL = True
USE_BLACKBOARD = True
USE_MULTI_AGENT = False   # nyalakan nanti
USE_META_EPSILON = True

# -------------------- ENVIRONMENT ----------------
GRID_SIZE = 10
EMPTY, DIRT, BLOCK, AGENT = 0, 1, -1, 2

np.random.seed(1)
grid = np.zeros((GRID_SIZE, GRID_SIZE), dtype=int)

for _ in range(12):
    x, y = np.random.randint(0, GRID_SIZE, 2)
    grid[x, y] = DIRT

for _ in range(10):
    x, y = np.random.randint(0, GRID_SIZE, 2)
    if grid[x, y] == EMPTY:
        grid[x, y] = BLOCK

agent_pos = [0, 0]

ACTIONS = {
    0: (-1, 0),
    1: (1, 0),
    2: (0, -1),
    3: (0, 1)
}

# -------------------- PERCEPTION -----------------
def perceive(grid, pos):
    obs = []
    for dx, dy in ACTIONS.values():
        nx, ny = pos[0]+dx, pos[1]+dy
        if nx < 0 or ny < 0 or nx >= GRID_SIZE or ny >= GRID_SIZE:
            obs.append(BLOCK)
        else:
            obs.append(grid[nx, ny])
    return np.array(obs)

# -------------------- SUPERVISED BRAIN -----------
X_train, y_train = [], []
for _ in range(800):
    obs = np.random.choice([EMPTY, DIRT, BLOCK], 4)
    if DIRT in obs:
        act = int(np.where(obs == DIRT)[0][0])
    else:
        safe = [i for i,v in enumerate(obs) if v != BLOCK]
        act = np.random.choice(safe) if safe else 0
    X_train.append(obs)
    y_train.append(act)

brain = DecisionTreeClassifier(max_depth=5)
brain.fit(X_train, y_train)

# -------------------- MEMORY LAYERS ---------------
VISIT_MEMORY = deque(maxlen=20)

BLACKBOARD = {
    "known_dirt": set(),
    "paths": set()
}

# -------------------- RL MEMORY -------------------
Q = defaultdict(lambda: np.zeros(4))
ALPHA, GAMMA = 0.1, 0.9

EPSILON = 0.3
EPSILON_MIN = 0.05
EPSILON_DECAY = 0.995

# -------------------- STEP FUNCTION ---------------
def step(grid, pos):
    global EPSILON
    obs = perceive(grid, pos)
    state = tuple(pos + obs.tolist())
    VISIT_MEMORY.append(state)

    deadlock = VISIT_MEMORY.count(state) > 2

    # ---------- ACTION SELECTION ----------
    if deadlock or np.random.rand() < EPSILON:
        actions = list(ACTIONS.keys())
        action = np.random.choice(actions)
    else:
        action = int(brain.predict([obs])[0])

    dx, dy = ACTIONS[action]
    nx, ny = pos[0]+dx, pos[1]+dy

    reward = -0.01
    if nx < 0 or ny < 0 or nx >= GRID_SIZE or ny >= GRID_SIZE:
        reward = -0.2
        nx, ny = pos
    elif grid[nx, ny] == BLOCK:
        reward = -0.2
        nx, ny = pos
    elif grid[nx, ny] == DIRT:
        reward = 1.0
        grid[nx, ny] = EMPTY

    new_state = tuple([nx, ny] + perceive(grid, [nx, ny]).tolist())

    # ---------- RL UPDATE ----------
    if USE_RL:
        Q[state][action] += ALPHA * (
            reward + GAMMA * np.max(Q[new_state]) - Q[state][action]
        )

    # ---------- BLACKBOARD ----------
    if USE_BLACKBOARD:
        BLACKBOARD["paths"].add((nx, ny))
        if reward > 0:
            BLACKBOARD["known_dirt"].add((nx, ny))

    # ---------- META-EPSILON ----------
    if USE_META_EPSILON:
        if reward > 0:
            EPSILON = max(EPSILON_MIN, EPSILON * EPSILON_DECAY)
        else:
            EPSILON = min(0.5, EPSILON / EPSILON_DECAY)

    return [nx, ny]

# -------------------- ANIMATION -------------------
fig, ax = plt.subplots(figsize=(5,5))
im = ax.imshow(grid, cmap="viridis", vmin=-1, vmax=2)

def update(frame):
    global agent_pos
    agent_pos = step(grid, agent_pos)
    display = grid.copy()
    display[agent_pos[0], agent_pos[1]] = AGENT
    im.set_data(display)
    ax.set_title(f"Step {frame} | ε={EPSILON:.2f}")
    return [im]

ani = animation.FuncAnimation(fig, update, frames=120, interval=300)
plt.close()
HTML(ani.to_jshtml())
